# ZIT Bag — PP + ZIT HP Joint HPO (temp)

목표: 기존 `bag_zit_hpo`(PP 고정 + HP 탐색)와 `bag_zit_pp_hpo`(PP 탐색 + HP 고정)를 합쳐,
전처리 파라미터와 ZIT 모델 파라미터를 한 Optuna study에서 함께 탐색한다.

- 모듈 파일은 수정하지 않고 import만 사용한다.
- Bag constraint는 기존 temp 노트북처럼 inline subclass override로 유지한다.
- Optuna sampler는 `TPESampler(multivariate=True, group=True)`로 PP/HP 조합을 같이 학습한다.


## 1. 환경 + import

In [ ]:
import os, sys

%run ../../../setup.py

import io, contextlib
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess
from modules.zi_tweedie import ZITboostRegressor

import lightgbm as lgb
from sklearn.model_selection import KFold
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

## 2. 실험 설정 (PP + ZIT HP joint)


In [ ]:
# ── 실험 식별 ──
EXP_ID  = 'zit-bag-joint-hpo-001'
USER    = 'jh'
N_TRIALS = 30
N_FOLDS  = 5
N_STARTUP_TRIALS = 8
MIN_FEATURES = 50
CLIP_Y_EXTREME = True

# ── tau_pi 학습 ──
USE_DIE_PI_THRESHOLD = True

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, '_temp', 'zit_bag_joint_hpo')
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')

MODEL_LABEL = 'ZIT Bag (PP + ZIT HP joint HPO)'
UNIT_AGGREGATION = 'sum'

print(f'EXP_ID={EXP_ID} | USER={USER} | N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS}')
print(f'N_STARTUP_TRIALS={N_STARTUP_TRIALS} | MIN_FEATURES={MIN_FEATURES}')
print(f'USE_DIE_PI_THRESHOLD={USE_DIE_PI_THRESHOLD}')
print(f'OUT_DIR={OUT_DIR}')
print(f'DB_PATH={DB_PATH}')


## 3. 데이터 로드 (전처리는 매 trial 다시 적용)


In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

print(f'\n[데이터 로드 완료]')
print(f'  xs: {xs.shape}, feat_cols: {len(feat_cols)}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')

## 4. BagZITboostRegressor 정의 (inline override)


In [ ]:
class BagZITboostRegressor(ZITboostRegressor):
    """ZITboost + bag (unit) constraint via B3 allocation."""

    @staticmethod
    def _allocate_b3(unit_y_per_unit, contribution, inverse, n_units):
        contrib_sum_per_unit = np.zeros(n_units)
        np.add.at(contrib_sum_per_unit, inverse, contribution)
        contrib_sum_die = contrib_sum_per_unit[inverse]
        n_die_per_unit = np.bincount(inverse, minlength=n_units).astype(np.float64)
        n_die_die = n_die_per_unit[inverse]
        share = np.where(
            contrib_sum_die > 1e-12,
            contribution / np.maximum(contrib_sum_die, 1e-12),
            1.0 / np.maximum(n_die_die, 1.0),
        )
        return unit_y_per_unit[inverse] * share

    def fit(self, X, y, unit_id):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).ravel()
        unit_id = np.asarray(unit_id)

        unique_units, first_idx, inverse = np.unique(
            unit_id, return_index=True, return_inverse=True
        )
        n_units = len(unique_units)
        unit_y_per_unit = y[first_idx]

        n_die_per_unit = np.bincount(inverse, minlength=n_units).astype(np.float64)
        y_die_alloc = unit_y_per_unit[inverse] / np.maximum(n_die_per_unit[inverse], 1.0)

        pi_arr, mu_arr, phi_arr = self._initialize(X, y_die_alloc)
        self._phi_current = phi_arr

        self.em_history_ = []
        prev_rmse = np.inf
        em_iter = 0

        for em_iter in range(self.n_em_iters):
            if em_iter > 0:
                contribution = np.clip((1 - pi_arr) * mu_arr, 0, None)
                y_die_alloc = self._allocate_b3(
                    unit_y_per_unit, contribution, inverse, n_units
                )

            posterior = self._e_step(y_die_alloc, pi_arr, mu_arr, phi_arr)

            self._phi_current = phi_arr
            lgb_pi, lgb_mu, lgb_phi, pi_arr, mu_arr, phi_arr = \
                self._m_step(X, y_die_alloc, posterior)

            pred_die = np.clip((1 - pi_arr) * mu_arr, 0, None)
            pred_unit = np.zeros(n_units)
            np.add.at(pred_unit, inverse, pred_die)
            rmse_unit = float(np.sqrt(np.mean((unit_y_per_unit - pred_unit) ** 2)))

            self.em_history_.append({
                'iter':      em_iter + 1,
                'unit_rmse': rmse_unit,
                'pi_mean':   float(pi_arr.mean()),
                'mu_mean':   float(mu_arr.mean()),
            })

            rmse_delta = prev_rmse - rmse_unit
            if em_iter >= 2 and abs(rmse_delta) < self.em_tol:
                break
            prev_rmse = rmse_unit

        self.n_em_iters_actual_ = em_iter + 1
        self.lgb_pi_ = lgb_pi
        self.lgb_mu_ = lgb_mu
        self.lgb_phi_ = lgb_phi
        self.fitted_ = True
        return self

    def predict_unit(self, X, unit_id):
        if not hasattr(self, 'fitted_'):
            raise ValueError('Model not fitted')
        unit_id = np.asarray(unit_id)
        unique_units, inverse = np.unique(unit_id, return_inverse=True)
        n_units = len(unique_units)
        pred_die = self.predict(X)
        pred_unit = np.zeros(n_units)
        np.add.at(pred_unit, inverse, pred_die)
        return pred_unit, unique_units


print('BagZITboostRegressor 정의 완료')

## 5. PP + ZIT HP 탐색공간 정의


In [ ]:
PP_PARAM_KEYS = [
    'missing_threshold',
    'corr_threshold',
    'add_indicator',
    'indicator_threshold',
    'spatial_max_dist',
    'post_impute_corr_threshold',
]
MODEL_PARAM_KEYS = [
    'zeta', 'n_em_iters',
    'mu_n_estimators', 'mu_learning_rate', 'mu_num_leaves', 'mu_max_depth',
    'mu_min_child_samples', 'mu_subsample', 'mu_colsample_bytree',
    'mu_reg_alpha', 'mu_reg_lambda',
    'pi_n_estimators', 'pi_learning_rate', 'pi_num_leaves', 'pi_max_depth',
    'pi_min_child_samples',
    'phi_n_estimators', 'phi_learning_rate', 'phi_num_leaves', 'phi_max_depth',
    'phi_min_child_samples',
]

def suggest_pp_params(trial):
    """전처리 PARAMS 탐색공간.

    Returns
    -------
    pp_params : dict
    """
    pp_params = dict(
        missing_threshold          = trial.suggest_float('missing_threshold', 0.20, 0.60),
        corr_threshold             = trial.suggest_float('corr_threshold', 0.85, 0.99),
        corr_keep_by               = 'std',                                                    # ★ 고정 (fold leakage 방지)
        add_indicator              = trial.suggest_categorical('add_indicator', [True, False]),
        indicator_threshold        = trial.suggest_float('indicator_threshold', 0.01, 0.20),
        spatial_max_dist           = trial.suggest_categorical('spatial_max_dist', [2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0]),
        post_impute_corr_threshold = trial.suggest_float('post_impute_corr_threshold', 0.95, 0.999),
        post_impute_corr_keep_by   = 'std',                                                    # ★ 고정
    )

    return pp_params


def suggest_zit_params(trial):
    """ZIT top 20 anchor 기반 narrow 탐색공간 .

    Returns
    -------
    model_params : dict
        model_params 는 ZIT 계열 모델에 전달.
    """
    params = dict(
        # ZIT 전용
        zeta=trial.suggest_float('zeta', 1.05, 1.50),
        n_em_iters=trial.suggest_int('n_em_iters', 8, 25),
        em_tol=1e-7,
        # μ (9개)
        mu_n_estimators=trial.suggest_int('mu_n_estimators', 100, 500),
        mu_learning_rate=trial.suggest_float('mu_learning_rate', 0.003, 0.03, log=True),
        mu_num_leaves=trial.suggest_int('mu_num_leaves', 60, 256),
        mu_max_depth=trial.suggest_int('mu_max_depth', 3, 8),
        mu_min_child_samples=trial.suggest_int('mu_min_child_samples', 20, 150),
        mu_subsample=trial.suggest_float('mu_subsample', 0.50, 0.95),
        mu_colsample_bytree=trial.suggest_float('mu_colsample_bytree', 0.20, 0.70),
        mu_reg_alpha=trial.suggest_float('mu_reg_alpha', 1e-6, 1e-1, log=True),
        mu_reg_lambda=trial.suggest_float('mu_reg_lambda', 1e-3, 1.0, log=True),
        # π (5개)
        pi_n_estimators=trial.suggest_int('pi_n_estimators', 100, 500),
        pi_learning_rate=trial.suggest_float('pi_learning_rate', 0.02, 0.20, log=True),
        pi_num_leaves=trial.suggest_int('pi_num_leaves', 30, 200),
        pi_max_depth=trial.suggest_int('pi_max_depth', 5, 12),
        pi_min_child_samples=trial.suggest_int('pi_min_child_samples', 10, 100),
        # φ (5개)
        phi_n_estimators=trial.suggest_int('phi_n_estimators', 30, 200),
        phi_learning_rate=trial.suggest_float('phi_learning_rate', 0.005, 0.05, log=True),
        phi_num_leaves=trial.suggest_int('phi_num_leaves', 30, 150),
        phi_max_depth=trial.suggest_int('phi_max_depth', 3, 8),
        phi_min_child_samples=trial.suggest_int('phi_min_child_samples', 10, 200),
        # 공통
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
        device='cpu',
    )

    return params


def suggest_joint_params(trial):
    pp_params = suggest_pp_params(trial)
    model_params = suggest_zit_params(trial)

    if USE_DIE_PI_THRESHOLD:
        tau_pi = trial.suggest_float('tau_pi', 0.5, 1.0)
    else:
        tau_pi = 1.0

    return pp_params, model_params, tau_pi


N_SEARCH_PARAMS = len(PP_PARAM_KEYS) + len(MODEL_PARAM_KEYS) + int(USE_DIE_PI_THRESHOLD)
print(f'Joint 탐색공간 정의 완료: PP={len(PP_PARAM_KEYS)}, ZIT={len(MODEL_PARAM_KEYS)}, '
      f'tau_pi={int(USE_DIE_PI_THRESHOLD)} -> total={N_SEARCH_PARAMS} HP')


## 6. K-fold objective (preprocess inside trial)


In [ ]:
import time

unit_ids_train_unique = y_train_unit.index.values
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(unit_ids_train_unique))


def _sum_die_to_unit(pred_die, uid_die):
    """die-level pred 를 unit별 합산."""
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    n_units = len(unique_units)
    pred_unit = np.zeros(n_units)
    np.add.at(pred_unit, inverse, pred_die)
    return pred_unit, unique_units


def _apply_tau_pi(pred_die, pi_die, tau_pi):
    return np.where(pi_die > tau_pi, 0.0, pred_die)


def _run_preprocess_silenced(pp_params):
    """preprocess.run() 출력 silenced. (pp_dict, n_feats) 반환."""
    with contextlib.redirect_stdout(io.StringIO()):
        pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=pp_params)
    return pp


def objective(trial):
    pp_params, model_params, tau_pi = suggest_joint_params(trial)

    t0 = time.time()

    # ── 전처리 (silenced) ──
    try:
        pp = _run_preprocess_silenced(pp_params)
    except Exception as e:
        print(f'  trial #{trial.number}: preprocess 실패 — {e}')
        raise optuna.TrialPruned()

    feat_cols_clean = pp['feat_cols']
    if len(feat_cols_clean) < 50:
        print(f'  trial #{trial.number}: feat_cols={len(feat_cols_clean)} 너무 적음 — pruned')
        trial.set_user_attr('pruned_reason', 'too_few_features')
        trial.set_user_attr('n_feats', len(feat_cols_clean))
        raise optuna.TrialPruned()

    xs_train_die = pp['xs_train']
    xs_val_die   = pp['xs_val']
    xs_test_die  = pp['xs_test']

    # numpy 변환
    X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
    X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
    X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
    uid_train_die = xs_train_die[KEY_COL].values
    uid_val_die   = xs_val_die[KEY_COL].values
    uid_test_die  = xs_test_die[KEY_COL].values
    y_train_die_broadcast = xs_train_die[KEY_COL].map(y_train_unit).values.astype(np.float64)

    pp_time = time.time() - t0

    # ── 5-fold 학습 (trial별 ZIT HP) ──
    oof_unit_pred  = pd.Series(np.nan, index=y_train_unit.index)
    fold_preds_val = []
    fold_preds_test = []
    fold_oof_rmse  = []

    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unit_ids_train_unique[tr_uidx]
        vl_units = unit_ids_train_unique[vl_uidx]
        tr_die_mask = np.isin(uid_train_die, tr_units)
        vl_die_mask = np.isin(uid_train_die, vl_units)

        X_tr   = X_train_die[tr_die_mask]
        X_vl   = X_train_die[vl_die_mask]
        y_tr   = y_train_die_broadcast[tr_die_mask]
        uid_tr = uid_train_die[tr_die_mask]
        uid_vl = uid_train_die[vl_die_mask]

        model = BagZITboostRegressor(**model_params)
        model.fit(X_tr, y_tr, unit_id=uid_tr)

        # OOF — die-level π/μ → τ_π 적용 → unit sum
        pi_vl, mu_vl, _ = model.predict_components(X_vl)
        pred_die_vl_raw = np.clip((1 - pi_vl) * mu_vl, 0, None)
        pred_die_vl = _apply_tau_pi(pred_die_vl_raw, pi_vl, tau_pi)
        pred_unit_vl, ids_vl = _sum_die_to_unit(pred_die_vl, uid_vl)

        oof_unit_pred.loc[ids_vl] = pred_unit_vl
        y_vl_true = y_train_unit.loc[ids_vl].values
        fold_rmse = float(np.sqrt(np.mean((pred_unit_vl - y_vl_true) ** 2)))
        fold_oof_rmse.append(fold_rmse)

        # holdout val/test
        pi_v, mu_v, _ = model.predict_components(X_val_die)
        pred_die_v_raw = np.clip((1 - pi_v) * mu_v, 0, None)
        pred_die_v = _apply_tau_pi(pred_die_v_raw, pi_v, tau_pi)
        pred_unit_val, ids_val = _sum_die_to_unit(pred_die_v, uid_val_die)

        pi_t, mu_t, _ = model.predict_components(X_test_die)
        pred_die_t_raw = np.clip((1 - pi_t) * mu_t, 0, None)
        pred_die_t = _apply_tau_pi(pred_die_t_raw, pi_t, tau_pi)
        pred_unit_test, ids_test = _sum_die_to_unit(pred_die_t, uid_test_die)

        fold_preds_val.append(pd.Series(pred_unit_val, index=ids_val))
        fold_preds_test.append(pd.Series(pred_unit_test, index=ids_test))

        # Pruning report
        avg_so_far = float(np.mean(fold_oof_rmse))
        trial.report(avg_so_far, step=fold_idx)
        if trial.should_prune():
            elapsed = time.time() - t0
            trial.set_user_attr('pruned_at_fold', fold_idx + 1)
            trial.set_user_attr('elapsed_sec', elapsed)
            trial.set_user_attr('tau_pi', tau_pi)
            trial.set_user_attr('n_feats', len(feat_cols_clean))
            raise optuna.TrialPruned()

    val_unit_pred  = pd.concat(fold_preds_val,  axis=1).mean(axis=1).reindex(y_val_unit.index)
    test_unit_pred = pd.concat(fold_preds_test, axis=1).mean(axis=1).reindex(y_test_unit.index)

    if oof_unit_pred.isna().any():
        raise RuntimeError('OOF NaN')

    oof_rmse  = float(np.sqrt(np.mean((oof_unit_pred.values  - y_train_unit.values) ** 2)))
    val_rmse  = float(np.sqrt(np.mean((val_unit_pred.values  - y_val_unit.values)  ** 2)))
    test_rmse = float(np.sqrt(np.mean((test_unit_pred.values - y_test_unit.values) ** 2)))

    elapsed = time.time() - t0
    trial.set_user_attr('val_rmse',     val_rmse)
    trial.set_user_attr('test_rmse',    test_rmse)
    trial.set_user_attr('fold_oof_rmse', fold_oof_rmse)
    trial.set_user_attr('elapsed_sec',  elapsed)
    trial.set_user_attr('pp_time_sec',  pp_time)
    trial.set_user_attr('tau_pi',       tau_pi)
    trial.set_user_attr('n_feats',      len(feat_cols_clean))

    print(f'  trial #{trial.number}: τ_π={tau_pi:.3f}, n_feats={len(feat_cols_clean)}, '
          f'oof={oof_rmse:.6f}, val={val_rmse:.6f}, test={test_rmse:.6f}, '
          f'pp={pp_time:.0f}s+fit={elapsed-pp_time:.0f}s')

    return oof_rmse


print(f'fold split 고정: {N_FOLDS} folds, {len(FOLDS)} 개')


## 7. Optuna study 실행 (grouped multivariate TPE)


In [ ]:
sampler = TPESampler(
    multivariate=True,
    group=True,
    n_startup_trials=N_STARTUP_TRIALS,
    seed=SEED,
)
pruner = MedianPruner(
    n_startup_trials=N_STARTUP_TRIALS,
    n_warmup_steps=2,
)

study_meta = {
    'exp_id':              EXP_ID,
    'user':                USER,
    'model':               MODEL_LABEL,
    'n_trials':            N_TRIALS,
    'n_folds':             N_FOLDS,
    'min_features':        MIN_FEATURES,
    'unit_aggregation':    UNIT_AGGREGATION,
    'use_die_pi_threshold': USE_DIE_PI_THRESHOLD,
    'sampler':             f'TPE multivariate group=True n_startup={N_STARTUP_TRIALS}',
    'pruner':              f'MedianPruner n_startup={N_STARTUP_TRIALS} n_warmup=2',
    'search_space':        'PP params + ZIT params + tau_pi joint',
    'CLIP_Y_EXTREME':      CLIP_Y_EXTREME,
    'SEED':                int(SEED),
}

study = optuna.create_study(
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    sampler=sampler,
    pruner=pruner,
    direction='minimize',
    load_if_exists=True,
)
for k, v in study_meta.items():
    study.set_user_attr(k, str(v))

print(f'study: {study.study_name}, DB: {DB_PATH}')
print(f'기존 trial 수: {len(study.trials)}')
print(f'sampler: TPESampler(multivariate=True, group=True, n_startup_trials={N_STARTUP_TRIALS})')

t_total = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'\n[HPO 완료] 전체 {time.time()-t_total:.0f}s, total trials={len(study.trials)}')
print(f'  best OOF RMSE: {study.best_value:.6f}')


## 8. Best params + trial history

In [ ]:
import matplotlib.pyplot as plt

best_trial = study.best_trial
best_params = best_trial.params

best_pp_preview = {k: best_params[k] for k in PP_PARAM_KEYS if k in best_params}
best_pp_preview['corr_keep_by'] = 'std'
best_pp_preview['post_impute_corr_keep_by'] = 'std'
best_model_preview = {k: best_params[k] for k in MODEL_PARAM_KEYS if k in best_params}

print(f'=== Best Trial #{best_trial.number} ===')
print(f'  OOF RMSE    : {best_trial.value:.6f}')
print(f'  val RMSE    : {best_trial.user_attrs.get("val_rmse", "N/A")}')
print(f'  test RMSE   : {best_trial.user_attrs.get("test_rmse", "N/A")}')
print(f'  best tau_pi : {best_trial.user_attrs.get("tau_pi", "N/A")}')
print(f'  n_feats     : {best_trial.user_attrs.get("n_feats", "N/A")}')
print(f'  elapsed     : {best_trial.user_attrs.get("elapsed_sec", 0):.0f}s')

print(f'\n  best PP params:')
for k, v in sorted(best_pp_preview.items()):
    print(f'    {k}: {v}')

print(f'\n  best ZIT params:')
for k, v in sorted(best_model_preview.items()):
    print(f'    {k}: {v}')

df_trials = pd.DataFrame([
    {
        'trial':     t.number,
        'state':     t.state.name,
        'oof_rmse':  t.value,
        'val_rmse':  t.user_attrs.get('val_rmse'),
        'test_rmse': t.user_attrs.get('test_rmse'),
        'tau_pi':    t.user_attrs.get('tau_pi'),
        'n_feats':   t.user_attrs.get('n_feats'),
        'elapsed':   t.user_attrs.get('elapsed_sec'),
    }
    for t in study.trials
])
print('\n=== trial history ===')
print(df_trials.to_string(index=False))

complete = df_trials[df_trials['state'] == 'COMPLETE']
if len(complete) > 1:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(complete['trial'], complete['oof_rmse'], marker='o', label='OOF')
    if complete['val_rmse'].notna().any():
        ax.plot(complete['trial'], complete['val_rmse'], marker='s', alpha=0.7, label='val')
    ax.axhline(study.best_value, color='red', linestyle='--', alpha=0.5, label=f'best={study.best_value:.6f}')
    ax.set_xlabel('trial')
    ax.set_ylabel('unit RMSE')
    ax.set_title(f'{MODEL_LABEL} joint HPO trial history')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


## 8.5 HP importance + 그룹별 분석


In [ ]:
import optuna.importance as opt_imp

try:
    imp = opt_imp.get_param_importances(study)

    print('=== HP importance (fANOVA) ===')
    for i, (k, v) in enumerate(sorted(imp.items(), key=lambda x: -x[1])):
        bar = '#' * int(v * 50)
        print(f'  {i+1:2d}. {k:34s}: {v:.4f}  {bar}')

    groups = {
        'PP_missing_indicator': 0.0,
        'PP_corr':             0.0,
        'PP_spatial':          0.0,
        'ZIT_core':            0.0,
        'ZIT_mu':              0.0,
        'ZIT_pi':              0.0,
        'ZIT_phi':             0.0,
        'tau_pi':              0.0,
    }
    for k, v in imp.items():
        if k in ('missing_threshold', 'add_indicator', 'indicator_threshold'):
            groups['PP_missing_indicator'] += v
        elif k in ('corr_threshold', 'post_impute_corr_threshold'):
            groups['PP_corr'] += v
        elif k == 'spatial_max_dist':
            groups['PP_spatial'] += v
        elif k in ('zeta', 'n_em_iters'):
            groups['ZIT_core'] += v
        elif k.startswith('mu_'):
            groups['ZIT_mu'] += v
        elif k.startswith('pi_'):
            groups['ZIT_pi'] += v
        elif k.startswith('phi_'):
            groups['ZIT_phi'] += v
        elif k == 'tau_pi':
            groups['tau_pi'] += v

    print('\n=== 그룹별 importance 합산 ===')
    for g, v in sorted(groups.items(), key=lambda x: -x[1]):
        bar = '#' * int(v * 50)
        print(f'  {g:22s}: {v:.4f}  {bar}')

    try:
        import optuna.visualization as ov
        ov.plot_param_importances(study).show()
        ov.plot_parallel_coordinate(study).show()
        top2 = list(sorted(imp.items(), key=lambda x: -x[1]))[:2]
        if len(top2) >= 2:
            ov.plot_contour(study, params=[top2[0][0], top2[1][0]]).show()
    except Exception as e:
        print(f'\nplotly 시각화 스킵: {e}')

except (RuntimeError, ValueError) as e:
    print(f'importance 분석 스킵: {e}')
    print('  -> trial 수가 적거나 모든 trial 이 같은 값일 가능성.')


## 9. Best PP + ZIT HP 5-fold refit + die-level 캡처


In [ ]:
best_oof_rmse  = float(best_trial.value)
best_val_rmse  = float(best_trial.user_attrs.get('val_rmse', np.nan))
best_test_rmse = float(best_trial.user_attrs.get('test_rmse', np.nan))
best_tau_pi    = float(best_trial.user_attrs.get('tau_pi', 1.0))

print('=== Best (5-fold OOF + holdout) ===')
print(f'  OOF unit RMSE  : {best_oof_rmse:.6f}')
print(f'  val unit RMSE  : {best_val_rmse:.6f}')
print(f'  test unit RMSE : {best_test_rmse:.6f}')
print(f'  best τ_π       : {best_tau_pi:.4f}')

# best PP params 만 추출 (tau_pi 제외)
best_pp_params = {k: v for k, v in best_params.items() if k != 'tau_pi'}
best_pp_params['corr_keep_by'] = 'std'              # 고정값
best_pp_params['post_impute_corr_keep_by'] = 'std'  # 고정값

best_model_params = {k: best_params[k] for k in MODEL_PARAM_KEYS if k in best_params}
best_full_params = dict(
    **best_model_params,
    em_tol=1e-7,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
    device='cpu',
)

print(f'\n=== best PP params 로 preprocess 재실행 ===')
for k, v in sorted(best_pp_params.items()):
    print(f'  {k}: {v}')

pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=best_pp_params)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values
y_train_die_broadcast = xs_train_die[KEY_COL].map(y_train_unit).values.astype(np.float64)

print(f'\n  best 전처리 후 feat_cols: {len(feat_cols_clean)}')

# die-level 캡처용
n_train_die = len(X_train_die)
n_val_die   = len(X_val_die)
n_test_die  = len(X_test_die)

oof_die_pi   = np.full(n_train_die, np.nan)
oof_die_mu   = np.full(n_train_die, np.nan)
oof_die_pred = np.full(n_train_die, np.nan)

val_die_pi   = np.zeros(n_val_die)
val_die_mu   = np.zeros(n_val_die)
val_die_pred = np.zeros(n_val_die)

test_die_pi   = np.zeros(n_test_die)
test_die_mu   = np.zeros(n_test_die)
test_die_pred = np.zeros(n_test_die)

print('\n=== best PP + best ZIT HP 로 5-fold refit (die-level 캡처) ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unit_ids_train_unique[tr_uidx]
    vl_units = unit_ids_train_unique[vl_uidx]
    tr_die_mask = np.isin(uid_train_die, tr_units)
    vl_die_mask = np.isin(uid_train_die, vl_units)
    X_tr, y_tr = X_train_die[tr_die_mask], y_train_die_broadcast[tr_die_mask]
    X_vl       = X_train_die[vl_die_mask]
    uid_tr     = uid_train_die[tr_die_mask]

    model = BagZITboostRegressor(**best_full_params)
    model.fit(X_tr, y_tr, unit_id=uid_tr)

    pi_vl, mu_vl, _ = model.predict_components(X_vl)
    pred_vl = np.clip((1 - pi_vl) * mu_vl, 0, None)
    oof_die_pi[vl_die_mask]   = pi_vl
    oof_die_mu[vl_die_mask]   = mu_vl
    oof_die_pred[vl_die_mask] = pred_vl

    pi_v, mu_v, _ = model.predict_components(X_val_die)
    pi_t, mu_t, _ = model.predict_components(X_test_die)
    val_die_pi   += pi_v / N_FOLDS
    val_die_mu   += mu_v / N_FOLDS
    val_die_pred += np.clip((1 - pi_v) * mu_v, 0, None) / N_FOLDS
    test_die_pi   += pi_t / N_FOLDS
    test_die_mu   += mu_t / N_FOLDS
    test_die_pred += np.clip((1 - pi_t) * mu_t, 0, None) / N_FOLDS

    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

assert not np.isnan(oof_die_pi).any(),   'OOF die π 미커버'
assert not np.isnan(oof_die_mu).any(),   'OOF die μ 미커버'
assert not np.isnan(oof_die_pred).any(), 'OOF die pred 미커버'

print('\n[refit 완료] die-level π/μ/pred 캡처 OK')
print(f'  oof_die: {n_train_die}, val_die: {n_val_die}, test_die: {n_test_die}')


## 10. tau_pi 적용 + raw vs clipped 비교


In [ ]:
oof_die_pred_clipped  = np.where(oof_die_pi  > best_tau_pi, 0.0, oof_die_pred)
val_die_pred_clipped  = np.where(val_die_pi  > best_tau_pi, 0.0, val_die_pred)
test_die_pred_clipped = np.where(test_die_pi > best_tau_pi, 0.0, test_die_pred)

oof_unit_raw_arr,  oof_unit_ids  = _sum_die_to_unit(oof_die_pred,         uid_train_die)
oof_unit_clip_arr, _              = _sum_die_to_unit(oof_die_pred_clipped, uid_train_die)
val_unit_raw_arr,  val_unit_ids  = _sum_die_to_unit(val_die_pred,         uid_val_die)
val_unit_clip_arr, _              = _sum_die_to_unit(val_die_pred_clipped, uid_val_die)
test_unit_raw_arr, test_unit_ids = _sum_die_to_unit(test_die_pred,        uid_test_die)
test_unit_clip_arr, _             = _sum_die_to_unit(test_die_pred_clipped, uid_test_die)

oof_unit_raw   = pd.Series(oof_unit_raw_arr,  index=oof_unit_ids).reindex(y_train_unit.index)
oof_unit_clip  = pd.Series(oof_unit_clip_arr, index=oof_unit_ids).reindex(y_train_unit.index)
val_unit_raw   = pd.Series(val_unit_raw_arr,  index=val_unit_ids).reindex(y_val_unit.index)
val_unit_clip  = pd.Series(val_unit_clip_arr, index=val_unit_ids).reindex(y_val_unit.index)
test_unit_raw  = pd.Series(test_unit_raw_arr, index=test_unit_ids).reindex(y_test_unit.index)
test_unit_clip = pd.Series(test_unit_clip_arr, index=test_unit_ids).reindex(y_test_unit.index)

def _rmse(pred, true):
    return float(np.sqrt(np.mean((pred.values - true.values) ** 2)))

oof_rmse_raw      = _rmse(oof_unit_raw,   y_train_unit)
oof_rmse_clipped  = _rmse(oof_unit_clip,  y_train_unit)
val_rmse_raw      = _rmse(val_unit_raw,   y_val_unit)
val_rmse_clipped  = _rmse(val_unit_clip,  y_val_unit)
test_rmse_raw     = _rmse(test_unit_raw,  y_test_unit)
test_rmse_clipped = _rmse(test_unit_clip, y_test_unit)

print('=' * 75)
print(f'  unit aggregation = sum | best tau_pi = {best_tau_pi:.4f}'
      + ('  (≈ off)' if best_tau_pi >= 0.99 else ''))
print('=' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"Raw":12s}  {oof_rmse_raw:11.6f}  {val_rmse_raw:11.6f}  {test_rmse_raw:11.6f}')
print(f'  {"Clipped":12s}  {oof_rmse_clipped:11.6f}  {val_rmse_clipped:11.6f}  {test_rmse_clipped:11.6f}')
print(f'  {"delta":12s}  {oof_rmse_clipped-oof_rmse_raw:+11.6f}  '
      f'{val_rmse_clipped-val_rmse_raw:+11.6f}  '
      f'{test_rmse_clipped-test_rmse_raw:+11.6f}')
print('=' * 75)

if val_rmse_clipped > val_rmse_raw and test_rmse_clipped > test_rmse_raw:
    print('  ⚠ val/test 모두 악화 — tau_pi 적용 비추천')
elif val_rmse_clipped < val_rmse_raw and test_rmse_clipped < test_rmse_raw:
    print('  ✅ val/test 모두 개선 — tau_pi 채택 권장')
else:
    print('  ⚠ val/test 결과 엇갈림 — 신중 판단')

killed_oof  = float((oof_die_pi  > best_tau_pi).mean())
killed_val  = float((val_die_pi  > best_tau_pi).mean())
killed_test = float((test_die_pi > best_tau_pi).mean())
print(f'\n  tau_pi 로 0 처리된 die 비율: OOF={killed_oof:.1%}, val={killed_val:.1%}, test={killed_test:.1%}')


## 11. 아티팩트 저장


In [ ]:
import json

def _build_die_df(uid_arr, die_id_arr, position_arr, pi, mu, pred, pred_clipped, y_unit):
    df = pd.DataFrame({
        KEY_COL:        uid_arr,
        DIE_KEY_COL:    die_id_arr,
        'position':     position_arr,
        'pi':           pi,
        'one_minus_pi': 1.0 - pi,
        'mu':           mu,
        'pred':         pred,
        'pred_clipped': pred_clipped,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df

oof_die_df = _build_die_df(
    uid_train_die, xs_train_die[DIE_KEY_COL].values, xs_train_die['position'].values,
    oof_die_pi, oof_die_mu, oof_die_pred, oof_die_pred_clipped, y_train_unit,
)
val_die_df = _build_die_df(
    uid_val_die, xs_val_die[DIE_KEY_COL].values, xs_val_die['position'].values,
    val_die_pi, val_die_mu, val_die_pred, val_die_pred_clipped, y_val_unit,
)
test_die_df = _build_die_df(
    uid_test_die, xs_test_die[DIE_KEY_COL].values, xs_test_die['position'].values,
    test_die_pi, test_die_mu, test_die_pred, test_die_pred_clipped, y_test_unit,
)
oof_die_df.to_csv(os.path.join(OUT_DIR,  'oof_die.csv'),  index=False)
val_die_df.to_csv(os.path.join(OUT_DIR,  'val_die.csv'),  index=False)
test_die_df.to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)

def _build_unit_df(unit_pred, y_unit):
    return pd.DataFrame({
        KEY_COL:  unit_pred.index.values,
        'pred':   unit_pred.values,
        'health': y_unit.reindex(unit_pred.index).values,
    })

_build_unit_df(oof_unit_raw,  y_train_unit).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(val_unit_raw,  y_val_unit  ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(test_unit_raw, y_test_unit ).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

_build_unit_df(oof_unit_clip,  y_train_unit).to_csv(os.path.join(OUT_DIR, 'oof_unit_clipped.csv'),  index=False)
_build_unit_df(val_unit_clip,  y_val_unit  ).to_csv(os.path.join(OUT_DIR, 'val_unit_clipped.csv'),  index=False)
_build_unit_df(test_unit_clip, y_test_unit ).to_csv(os.path.join(OUT_DIR, 'test_unit_clipped.csv'), index=False)

best_payload = {
    'best_trial_number': best_trial.number,
    'best_oof_rmse':     best_oof_rmse,
    'best_val_rmse':     best_val_rmse,
    'best_test_rmse':    best_test_rmse,
    'best_tau_pi':       best_tau_pi,
    'best_pp_params':    best_pp_params,
    'best_model_params': best_model_params,
    'best_model_params_resolved': best_full_params,
    'unit_aggregation':  UNIT_AGGREGATION,
    'n_features':        len(feat_cols_clean),
}
with open(os.path.join(OUT_DIR, 'best_params.json'), 'w', encoding='utf-8') as f:
    json.dump(best_payload, f, indent=2, ensure_ascii=False, default=str)

meta = {
    'exp_id':              EXP_ID,
    'model':               MODEL_LABEL,
    'unit_aggregation':    UNIT_AGGREGATION,
    'use_die_pi_threshold': USE_DIE_PI_THRESHOLD,
    'best_tau_pi':         best_tau_pi,
    'n_trials_run':        len(study.trials),
    'n_trials_complete':   sum(1 for t in study.trials if t.state.name == 'COMPLETE'),
    'n_folds':             N_FOLDS,
    'best_pp_params':      best_pp_params,
    'best_model_params':   best_model_params,
    'best_n_feats':        len(feat_cols_clean),
    'raw_oof_rmse':        oof_rmse_raw,
    'raw_val_rmse':        val_rmse_raw,
    'raw_test_rmse':       test_rmse_raw,
    'clipped_oof_rmse':    oof_rmse_clipped,
    'clipped_val_rmse':    val_rmse_clipped,
    'clipped_test_rmse':   test_rmse_clipped,
    'study_best_oof_rmse': best_oof_rmse,
    'study_best_val_rmse': best_val_rmse,
    'study_best_test_rmse': best_test_rmse,
    'CLIP_Y_EXTREME':      CLIP_Y_EXTREME,
    'SEED':                int(SEED),
    'sampler':             f'TPE multivariate group=True n_startup={N_STARTUP_TRIALS}',
    'tau_pi_range':        '[0.5, 1.0]' if USE_DIE_PI_THRESHOLD else 'off',
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:35s}  {sz:10,.1f} KB')


## 12. 요약


In [ ]:
print('=' * 75)
print(' ZIT Bag Joint HPO (PP + ZIT HP + tau_pi) — 결과 요약')
print('=' * 75)
print(f'  EXP_ID            : {EXP_ID}')
print(f'  N_TRIALS run      : {len(study.trials)}')
print(f'  완료 trial        : {sum(1 for t in study.trials if t.state.name == "COMPLETE")}')
print(f'  Best trial        : #{best_trial.number}')
print(f'  Best tau_pi       : {best_tau_pi:.4f}' + ('  (approx off)' if best_tau_pi >= 0.99 else ''))
print(f'  Best n_feats      : {len(feat_cols_clean)}')
print(f'  Sampler           : TPE multivariate group=True')
print('-' * 75)
print(f'  {"":10s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"Raw":10s}  {oof_rmse_raw:11.6f}  {val_rmse_raw:11.6f}  {test_rmse_raw:11.6f}')
print(f'  {"Clipped":10s}  {oof_rmse_clipped:11.6f}  {val_rmse_clipped:11.6f}  {test_rmse_clipped:11.6f}')
print(f'  {"delta":10s}  {oof_rmse_clipped-oof_rmse_raw:+11.6f}  '
      f'{val_rmse_clipped-val_rmse_raw:+11.6f}  '
      f'{test_rmse_clipped-test_rmse_raw:+11.6f}')
print('-' * 75)
print('  비교 기준선:')
print('    BagZIT baseline (HP+PP fixed):   OOF=0.005524, val=0.005729, test=0.008428')
print('    BagZIT PP HPO (HP fixed):        기존 bag_zit_pp_hpo 결과와 비교')
print('    BagZIT HP HPO (PP fixed):        기존 bag_zit_hpo 결과와 비교')
print('=' * 75)
